# Phase 01 — Crawling & Scraping Berita Ekonomi CNBC Indonesia

Notebook ini adalah **satu fase utuh** dari pipeline penelitian ABSA (lihat `ringkasan_penelitian_absa_berita_ekonomi.md`). Tujuannya:

1. **Crawling** — mengumpulkan daftar URL artikel ekonomi dari **sitemap** resmi CNBC Indonesia (bukan halaman indeks, karena indeks dirender JavaScript dan di-*disallow* di `robots.txt`).
2. **Scraping** — membuka tiap URL dan mengekstrak **judul, tanggal terbit, kanal, isi mentah, keywords**.

### Prinsip desain
- **Data mentah disimpan apa adanya.** Cleaning / pemecahan kalimat dilakukan di Phase 02, bukan di sini — supaya proses bisa diulang tanpa crawl ulang.
- **Checkpoint & resume.** URL hasil crawl disimpan ke `urls.csv`. Hasil scrape ditulis *append* baris-per-baris ke `articles.jsonl`. Kalau internet mati atau notebook dihentikan, jalankan ulang sel scraping — artikel yang sudah selesai akan dilewati, jadi **tidak mengulang dari nol**.
- **Sopan ke server.** Ada jeda antar-permintaan (`REQUEST_DELAY`), retry dengan backoff, dan User-Agent yang jelas. Query string (`?`) dan halaman indeks dihindari sesuai `robots.txt`.
- **Pilot dulu.** Set `PILOT = True` untuk menguji alur pada rentang pendek (beberapa hari) sebelum menjalankan full scrape 2021–2026.

> **Catatan hukum/etika:** `robots.txt` hanya aturan teknis. Pastikan tim juga membaca Ketentuan Penggunaan CNBC Indonesia. Data dipakai untuk riset akademik non-komersial.

## 1. Konfigurasi

Semua tombol pengaturan ada di sini. Yang paling sering diubah: `PILOT`, `PERIODE_MULAI`, `PERIODE_SELESAI`, `KANAL`.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone, timedelta

# --- Mode ---
# PILOT=True  -> hanya rentang pendek + batasi jumlah artikel, untuk uji alur.
# PILOT=False -> full crawl & scrape sesuai PERIODE di bawah.
PILOT = True

# --- Periode data yang diinginkan (zona waktu WIB, +07:00) ---
WIB = timezone(timedelta(hours=7))
PERIODE_MULAI   = datetime(2021, 1, 1, tzinfo=WIB)
PERIODE_SELESAI = datetime(2026, 10, 22, 23, 59, 59, tzinfo=WIB)  # per hari ini

# Untuk PILOT: rentang pendek (default 3 hari terakhir dari batas atas)
PILOT_MULAI   = datetime(2026, 10, 20, tzinfo=WIB)
PILOT_SELESAI = PERIODE_SELESAI
PILOT_MAX_ARTIKEL = 15   # batasi jumlah artikel yang di-scrape saat pilot

# --- Kanal ekonomi yang diambil ---
# 'market' = pasar modal/ekonomi (utama). Tambah 'research','syariah' bila perlu.
# Hindari kanal non-ekonomi (lifestyle, tech, dll) agar aspek BERTopic tetap bersih.
KANAL = ["market"]
if not PILOT:
    KANAL = ["market", "research", "syariah"]

# Jenis sitemap yang ditelusuri per kanal.
# sitemap_news = artikel terbaru; sitemap_web = arsip lebih dalam (dibutuhkan utk 2021).
SITEMAP_JENIS = ["sitemap_news.xml", "sitemap_web.xml"]
if PILOT:
    SITEMAP_JENIS = ["sitemap_news.xml"]  # cukup yang terbaru untuk pilot

# --- Jaringan (sopan ke server) ---
BASE = "https://www.cnbcindonesia.com"
SITEMAP_INDEX = f"{BASE}/sitemap.xml"
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "ABSA-Research-Crawler/1.0 (akademik; kontak: email-anda@contoh.ac.id)"
)
REQUEST_DELAY = 1.5      # jeda detik antar permintaan artikel
SITEMAP_DELAY = 1.0      # jeda detik antar permintaan sitemap
TIMEOUT = 20            # detik
MAX_RETRY = 3

# --- Lokasi penyimpanan ---
DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

suffix = "pilot" if PILOT else "full"
URLS_CSV      = RAW_DIR / f"urls_{suffix}.csv"          # checkpoint hasil crawling
ARTICLES_JSONL = RAW_DIR / f"articles_{suffix}.jsonl"   # hasil scrape (append per baris)
ARTICLES_CSV   = RAW_DIR / f"articles_{suffix}.csv"     # ekspor akhir (opsional)

print(f"Mode           : {'PILOT' if PILOT else 'FULL'}")
print(f"Periode efektif: {(PILOT_MULAI if PILOT else PERIODE_MULAI).date()} s/d "
      f"{(PILOT_SELESAI if PILOT else PERIODE_SELESAI).date()}")
print(f"Kanal          : {KANAL}")
print(f"Sitemap jenis  : {SITEMAP_JENIS}")
print(f"Output URL     : {URLS_CSV}")
print(f"Output artikel : {ARTICLES_JSONL}")

## 2. Utilitas jaringan

Satu `requests.Session` dengan retry/backoff manual. Fungsi `fetch()` dipakai untuk sitemap maupun artikel.

In [ ]:
import time
import requests

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT, "Accept-Language": "id,en;q=0.8"})

def fetch(url, timeout=TIMEOUT, max_retry=MAX_RETRY):
    """Unduh URL dengan retry + exponential backoff. Return response atau None."""
    url = url.strip()
    for percobaan in range(1, max_retry + 1):
        try:
            resp = session.get(url, timeout=timeout)
            if resp.status_code == 200:
                return resp
            # 404/410 = artikel hilang, tidak perlu retry
            if resp.status_code in (404, 410):
                print(f"    [{resp.status_code}] dilewati: {url}")
                return None
            print(f"    [{resp.status_code}] percobaan {percobaan}/{max_retry}: {url}")
        except requests.RequestException as e:
            print(f"    [ERR] percobaan {percobaan}/{max_retry}: {type(e).__name__} - {url}")
        time.sleep(2 ** percobaan)  # backoff: 2s, 4s, 8s
    return None

## 3. Crawling — kumpulkan URL artikel dari sitemap

Alur:
`sitemap.xml` (index) → sub-sitemap per kanal (`market/sitemap_news.xml`, dst) → daftar URL + metadata.

Dari tiap entri sitemap kita ambil: **URL, tanggal terbit, judul, keywords**. Lalu parsing pola URL untuk mendapatkan **id_artikel, kanal, kode tipe** (17=teks, 18/19=video).

In [ ]:
import re
from bs4 import BeautifulSoup

# Pola URL: /{kanal}/{YYYYMMDDhhmmss}-{tipe}-{id}/{slug}
URL_PATTERN = re.compile(r"/([a-z0-9\-]+)/(\d{14})-(\d+)-(\d+)/")

def parse_url_meta(url):
    """Ambil (kanal, timestamp_str, kode_tipe, id_artikel) dari pola URL. None jika tak cocok."""
    m = URL_PATTERN.search(url)
    if not m:
        return None
    kanal, ts, kode_tipe, id_artikel = m.groups()
    return {"kanal": kanal, "ts": ts, "kode_tipe": kode_tipe, "id_artikel": id_artikel}

def is_video(kode_tipe, judul):
    """Artikel video (kode 18/19 atau judul diawali 'Video:') dibuang: minim teks."""
    if kode_tipe in ("18", "19"):
        return True
    if judul and judul.strip().lower().startswith("video"):
        return True
    return False

def parse_tanggal(iso_str):
    """Parse '2026-10-22T20:40:00+07:00' -> datetime berzona. None jika gagal."""
    try:
        return datetime.fromisoformat(iso_str.strip())
    except (ValueError, AttributeError):
        return None

def ambil_entri_sitemap(xml_text):
    """Parse sebuah sitemap XML -> list dict {url, tanggal_terbit, judul, keywords}."""
    soup = BeautifulSoup(xml_text, "xml")  # butuh lxml
    entri = []
    for url_tag in soup.find_all("url"):
        loc = url_tag.find("loc")
        if not loc or not loc.text:
            continue
        url = loc.text.strip()
        judul_tag = url_tag.find("news:title") or url_tag.find("title")
        tgl_tag   = url_tag.find("news:publication_date") or url_tag.find("lastmod")
        kw_tag    = url_tag.find("news:keywords") or url_tag.find("keywords")
        entri.append({
            "url": url,
            "tanggal_terbit": (tgl_tag.text.strip() if tgl_tag and tgl_tag.text else ""),
            "judul": (judul_tag.text.strip() if judul_tag and judul_tag.text else ""),
            "keywords": (kw_tag.text.strip() if kw_tag and kw_tag.text else ""),
        })
    return entri

In [ ]:
import pandas as pd

def crawl_urls():
    """Telusuri sitemap -> kembalikan DataFrame URL artikel teks dalam rentang periode."""
    mulai   = PILOT_MULAI   if PILOT else PERIODE_MULAI
    selesai = PILOT_SELESAI if PILOT else PERIODE_SELESAI

    baris = []
    dilihat = set()  # dedupe berdasarkan id_artikel

    for kanal in KANAL:
        for jenis in SITEMAP_JENIS:
            sm_url = f"{BASE}/{kanal}/{jenis}"
            print(f"[CRAWL] {sm_url}")
            resp = fetch(sm_url)
            time.sleep(SITEMAP_DELAY)
            if resp is None:
                print("        (gagal / kosong, lanjut)")
                continue

            entri = ambil_entri_sitemap(resp.text)
            print(f"        {len(entri)} entri ditemukan")

            for e in entri:
                meta = parse_url_meta(e["url"])
                if meta is None:
                    continue
                if is_video(meta["kode_tipe"], e["judul"]):
                    continue

                # tanggal: pakai dari sitemap, fallback ke timestamp URL
                tgl = parse_tanggal(e["tanggal_terbit"])
                if tgl is None:
                    try:
                        tgl = datetime.strptime(meta["ts"], "%Y%m%d%H%M%S").replace(tzinfo=WIB)
                    except ValueError:
                        continue
                if not (mulai <= tgl <= selesai):
                    continue

                if meta["id_artikel"] in dilihat:
                    continue
                dilihat.add(meta["id_artikel"])

                baris.append({
                    "id_artikel": meta["id_artikel"],
                    "kanal": meta["kanal"],
                    "url": e["url"],
                    "tanggal_terbit": tgl.isoformat(),
                    "judul": e["judul"],
                    "keywords": e["keywords"],
                })

    df = pd.DataFrame(baris).drop_duplicates(subset="id_artikel").reset_index(drop=True)
    if PILOT and len(df) > PILOT_MAX_ARTIKEL:
        df = df.head(PILOT_MAX_ARTIKEL).copy()
        print(f"[PILOT] dibatasi {PILOT_MAX_ARTIKEL} artikel")
    return df

# Checkpoint: kalau urls.csv sudah ada, pakai itu (tidak crawl ulang). Hapus file untuk crawl ulang.
if URLS_CSV.exists():
    df_urls = pd.read_csv(URLS_CSV, dtype={"id_artikel": str})
    print(f"[SKIP CRAWL] memuat {len(df_urls)} URL dari checkpoint {URLS_CSV}")
else:
    df_urls = crawl_urls()
    df_urls.to_csv(URLS_CSV, index=False)
    print(f"[SIMPAN] {len(df_urls)} URL -> {URLS_CSV}")

df_urls.head(10)

## 4. Scraping — ekstrak isi tiap artikel (dengan resume)

Strategi ekstraksi ini **sudah diverifikasi** terhadap HTML artikel market CNBC asli (`test.html`), bukan tebakan:
- **Judul & tanggal** → dari **JSON-LD** blok `@type: NewsArticle` (`headline`, `datePublished`). Bersih dan andal. (Fallback: meta `og:`.)
- **Isi artikel** → dari kontainer **`div.detail-text`** (gabung paragraf `<p>`). *Catatan:* JSON-LD CNBC **tidak** memuat `articleBody`, jadi isi WAJIB diambil dari kontainer ini.

Noise yang dibuang di dalam kontainer (terverifikasi ada di HTML asli): `<script>` iklan (FMG_Tag/cads), `<div>` banner Google Ads (`div-gpt-ad...`), blok video sisipan (`a.autovideocnbc`, `.video_detail`), `.read-full-article`, kode penulis `(xxx/xxx)`, dan `<p>` kosong.

Noise (tautan 'baca juga', `[Gambas:Video ...]`, skrip) dibuang ringan, tapi **isi tetap disimpan mentah** — pembersihan mendalam & pemecahan kalimat ada di Phase 02.

**Resume:** tiap artikel yang berhasil ditulis sebagai satu baris JSON ke `articles.jsonl`. Saat sel dijalankan ulang, `id_artikel` yang sudah ada di file akan dilewati.

In [ ]:
import json

def _coerce_jsonld_list(data):
    """JSON-LD bisa berupa dict, list, atau punya '@graph'. Ratakan jadi list of dict."""
    out = []
    if isinstance(data, dict):
        if "@graph" in data and isinstance(data["@graph"], list):
            out.extend(data["@graph"])
        else:
            out.append(data)
    elif isinstance(data, list):
        out.extend(data)
    return out

def ekstrak_dari_jsonld(soup):
    """Cari NewsArticle/Article di JSON-LD. Return dict parsial {judul, tanggal}."""
    hasil = {}
    for tag in soup.find_all("script", type="application/ld+json"):
        raw = tag.string or tag.get_text() or ""
        if not raw.strip():
            continue
        try:
            data = json.loads(raw)
        except json.JSONDecodeError:
            continue
        for obj in _coerce_jsonld_list(data):
            if not isinstance(obj, dict):
                continue
            tipe = obj.get("@type", "")
            tipe_set = {tipe} if isinstance(tipe, str) else set(tipe or [])
            if tipe_set & {"NewsArticle", "Article", "ReportageNewsArticle"}:
                if obj.get("headline") and "judul" not in hasil:
                    hasil["judul"] = str(obj["headline"]).strip()
                if obj.get("datePublished") and "tanggal" not in hasil:
                    hasil["tanggal"] = str(obj["datePublished"]).strip()
                # CNBC TIDAK menyertakan articleBody di JSON-LD -> isi diambil dari kontainer.
    return hasil

def ekstrak_dari_kontainer(soup):
    """Ambil isi dari div.detail-text (struktur terverifikasi di test.html).
    Gabung paragraf <p>, setelah membuang iklan/script/video sisipan/kode penulis."""
    # Ada 2 lapis bersarang 'detail-text'; find() mengambil yang terluar,
    # dan .find_all('p') tetap menjangkau <p> di lapis dalam. Aman.
    node = soup.find("div", class_="detail-text")
    if node is None:
        return ""

    # 1) buang elemen non-teks di dalam kontainer
    for buang in node.find_all(["script", "style", "figure", "figcaption", "table", "iframe"]):
        buang.decompose()
    # 2) buang blok iklan/banner Google Ads (div dengan id div-gpt-ad...) + parallax
    for div in node.find_all("div", id=re.compile(r"div-gpt-ad|google_ads")):
        div.decompose()
    # 3) buang blok video sisipan (link autovideo + wrapper video_detail)
    for v in node.find_all("a", class_=re.compile(r"autovideocnbc")):
        v.decompose()
    for v in node.find_all("div", class_=re.compile(r"video_detail|sisip_embed")):
        v.decompose()
    # 4) buang placeholder 'read-full-article' dan tautan 'baca juga'
    for rf in node.find_all(class_=re.compile(r"read-full-article")):
        rf.decompose()
    for a in node.find_all("a"):
        teks_a = (a.get_text() or "").strip().lower()
        if teks_a.startswith("baca juga") or "selengkapnya" in teks_a:
            a.decompose()

    # 5) kumpulkan paragraf <p> yang tidak kosong
    paragraf = [p.get_text(" ", strip=True) for p in node.find_all("p")]
    paragraf = [p for p in paragraf if p and len(p) > 1]
    teks = "\n".join(paragraf).strip()

    # 6) buang kode penulis di akhir, mis. '(haa/haa)' / '(rob/rob)'
    teks = re.sub(r"\(\s*[a-z]{2,4}\s*/\s*[a-z]{2,4}\s*\)\s*$", "", teks).strip()
    return teks

def ekstrak_meta_og(soup, nama):
    tag = soup.find("meta", property=f"og:{nama}") or soup.find("meta", attrs={"name": nama})
    return tag["content"].strip() if tag and tag.get("content") else ""

def bersihkan_ringan(teks):
    """Pembersihan ringan SAJA (isi tetap mentah untuk Phase 02)."""
    if not teks:
        return ""
    teks = re.sub(r"\[Gambas:[^\]]*\]", " ", teks)   # buang penanda embed
    teks = re.sub(r"[ \t]+", " ", teks)
    teks = re.sub(r"\n{3,}", "\n\n", teks)
    return teks.strip()

def scrape_artikel(url):
    """Scrape satu artikel -> dict {judul, tanggal_terbit_html, isi, isi_panjang, metode, status}."""
    resp = fetch(url)
    if resp is None:
        return {"status": "gagal_unduh", "metode": "", "judul": "", "tanggal_terbit_html": "", "isi": "", "isi_panjang": 0}

    soup = BeautifulSoup(resp.text, "lxml")
    ld = ekstrak_dari_jsonld(soup)

    # ISI: utama dari kontainer div.detail-text (JSON-LD CNBC tidak punya articleBody)
    isi = ekstrak_dari_kontainer(soup)
    metode = "detail-text"

    # JUDUL & TANGGAL: utama dari JSON-LD, fallback ke meta og:
    judul = ld.get("judul") or ekstrak_meta_og(soup, "title") or (soup.title.get_text().strip() if soup.title else "")
    tanggal = ld.get("tanggal") or ekstrak_meta_og(soup, "article:published_time")
    isi = bersihkan_ringan(isi)

    status = "ok" if len(isi) >= 150 else "isi_kosong_atau_pendek"
    return {
        "status": status,
        "metode": metode,
        "judul": judul,
        "tanggal_terbit_html": tanggal,
        "isi": isi,
        "isi_panjang": len(isi),
    }

In [ ]:
def id_sudah_selesai(path):
    """Baca id_artikel yang sudah ada di JSONL -> set (untuk resume)."""
    selesai = set()
    if not path.exists():
        return selesai
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                selesai.add(str(json.loads(line)["id_artikel"]))
            except (json.JSONDecodeError, KeyError):
                continue
    return selesai

def jalankan_scrape(df_urls):
    """Loop scrape dengan resume: append tiap artikel ke JSONL segera setelah berhasil."""
    try:
        from tqdm.auto import tqdm
    except ImportError:  # tqdm opsional
        def tqdm(x, **k):
            return x

    selesai = id_sudah_selesai(ARTICLES_JSONL)
    print(f"[RESUME] {len(selesai)} artikel sudah ada, akan dilewati.")

    sisa = df_urls[~df_urls["id_artikel"].astype(str).isin(selesai)]
    print(f"[SCRAPE] {len(sisa)} artikel akan di-scrape.")

    n_ok = n_gagal = 0
    # buffering=1 -> line-buffered: tiap baris langsung ter-flush (aman bila mati mendadak)
    with open(ARTICLES_JSONL, "a", encoding="utf-8", buffering=1) as fout:
        try:
            for _, row in tqdm(sisa.iterrows(), total=len(sisa), desc="scraping"):
                hasil = scrape_artikel(row["url"])
                rekaman = {
                    "id_artikel": str(row["id_artikel"]),
                    "kanal": row["kanal"],
                    "url": row["url"],
                    "tanggal_terbit": row["tanggal_terbit"],
                    "judul_sitemap": row["judul"],
                    "keywords": row["keywords"],
                    "judul": hasil["judul"],
                    "tanggal_terbit_html": hasil["tanggal_terbit_html"],
                    "isi_mentah": hasil["isi"],
                    "isi_panjang": hasil["isi_panjang"],
                    "metode_ekstraksi": hasil["metode"],
                    "status": hasil["status"],
                    "tanggal_scrape": datetime.now(WIB).isoformat(),
                }
                fout.write(json.dumps(rekaman, ensure_ascii=False) + "\n")
                if hasil["status"] == "ok":
                    n_ok += 1
                else:
                    n_gagal += 1
                time.sleep(REQUEST_DELAY)
        except KeyboardInterrupt:
            print("\n[DIHENTIKAN] progres tersimpan. Jalankan ulang sel ini untuk melanjutkan.")

    print(f"[SELESAI] sesi ini: {n_ok} ok, {n_gagal} bermasalah.")

jalankan_scrape(df_urls)

## 5. Verifikasi hasil & ekspor

Muat `articles.jsonl`, cek jumlah, status, metode ekstraksi, dan lihat contoh isi. Lalu ekspor ke CSV sebagai salinan ringkas.

In [ ]:
df_art = pd.read_json(ARTICLES_JSONL, lines=True) if ARTICLES_JSONL.exists() else pd.DataFrame()
print(f"Total artikel ter-scrape : {len(df_art)}")
if len(df_art):
    print("\nDistribusi status:")
    print(df_art["status"].value_counts())
    print("\nMetode ekstraksi yang berhasil dipakai:")
    print(df_art["metode_ekstraksi"].value_counts())
    print(f"\nRata-rata panjang isi: {df_art['isi_panjang'].mean():.0f} karakter")
    print(f"Artikel dengan isi < 150 char: {(df_art['isi_panjang'] < 150).sum()}")
df_art[["id_artikel", "kanal", "tanggal_terbit", "status", "metode_ekstraksi", "isi_panjang", "judul"]].head(10)

In [ ]:
# Lihat satu contoh isi artikel yang berhasil (cek kualitas ekstraksi)
if len(df_art):
    ok = df_art[df_art["status"] == "ok"]
    if len(ok):
        contoh = ok.iloc[0]
        print("JUDUL :", contoh["judul"])
        print("TGL   :", contoh["tanggal_terbit"])
        print("URL   :", contoh["url"])
        print("METODE:", contoh["metode_ekstraksi"])
        print("-" * 70)
        print(contoh["isi_mentah"][:1200], "...")
    else:
        print("Belum ada artikel berstatus 'ok'.")

In [ ]:
# Ekspor salinan CSV (tanpa isi_mentah agar ringan; isi lengkap tetap di JSONL)
if len(df_art):
    kolom_csv = [c for c in df_art.columns if c != "isi_mentah"]
    df_art[kolom_csv].to_csv(ARTICLES_CSV, index=False)
    print(f"[SIMPAN] ringkasan -> {ARTICLES_CSV}")
    print("Isi lengkap (isi_mentah) tetap tersimpan di:", ARTICLES_JSONL)

## 6. Cara menjalankan full scrape (2021 – 22 Okt 2026)

Setelah pilot terbukti berjalan:

1. Di **sel Konfigurasi (bagian 1)**, ubah `PILOT = False`.
2. Jalankan ulang seluruh notebook dari atas (Restart & Run All).
3. Crawl akan menelusuri `sitemap_news.xml` **dan** `sitemap_web.xml` untuk kanal `market`, `research`, `syariah`.
4. Scrape berjalan dengan resume. Kalau koneksi putus / notebook ditutup, cukup jalankan ulang **sel bagian 4** — artikel yang sudah ada di `articles_full.jsonl` otomatis dilewati.

### Catatan penting untuk arsip 2021
`sitemap_web.xml` biasanya **tidak** memuat seluruh arsip bertahun-tahun; situs berita umumnya membatasi kedalaman sitemap. Kalau hasil crawl tidak mundur sampai 2021, ada dua opsi lanjutan (bisa kita bangun di iterasi berikut):
- Menelusuri sitemap arsip per bulan bila CNBC menyediakannya, atau
- Menggunakan arsip publik (mis. Wayback Machine CDX API) untuk menambang URL lama.

Pilot ini lebih dulu memastikan **mekanisme crawl + scrape + resume** bekerja. Cakupan arsip penuh ditangani setelah itu.